In [ ]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from scipy.stats import multivariate_normal, spearmanr
import metricas_plots

In [ ]:
importlib.reload(metricas_plots)
from metricas_plots import PlotsMetricas, T, F
p = PlotsMetricas()

### Ler dados

In [ ]:
dados = pd.read_csv("dados/ariel_limpo_log10.csv.gz", compression="gzip")

In [ ]:
# Definir feature, target e gerar splits
col_x = F.mass.value
df_metrics_bic = pd.read_csv(f"results/by_bic/best_popsizes_by_bic.csv")
larguras = p.targets[:4]
train, test = train_test_split(
    dados[[col_x] + larguras], test_size=0.3, random_state=4321
)

## Calcular Estatísticas por Bin para o treinamento

In [ ]:
# Faz o agrupamento dos dados
nbins = 150
train[f"{col_x}_mean"] = pd.qcut(train[col_x], nbins)
test[f"{col_x}_mean"] = pd.qcut(test[col_x], nbins)


def gera_stats(dados_split, col_x):
    stats_list = []
    for bin_name, group in dados_split.groupby(f"{col_x}_mean"):
        if len(group) < 5:  # Pular bins com poucos dados
            continue

        # Ponto central de cada bin
        col_center = group[col_x].mean()

        # Calcular média e desvio padrão
        dados_larguras = group[larguras].values
        means = dados_larguras.mean(axis=0)
        stds = dados_larguras.std(axis=0)

        # Calcular matriz de covariância 4x4
        cov_matrix = np.cov(dados_larguras, rowvar=False)

        # Montar dicionário com todas as estatísticas
        stat_dict = {col_x: col_center}
        for i, largura in enumerate(larguras):
            stat_dict[f"{largura}_mean"] = means[i]
            stat_dict[f"{largura}_std"] = stds[i]
        for l1, l2 in p.cov_pairs:  # covariâncias
            i1 = larguras.index(l1)
            i2 = larguras.index(l2)
            stat_dict[f"cov_{l1}_{l2}"] = cov_matrix[i1, i2]

        stats_list.append(stat_dict)
    return stats_list


train_por_bin = pd.DataFrame(gera_stats(train, col_x))
test_por_bin = pd.DataFrame(gera_stats(test, col_x))
test_por_bin.tail()

## Treinar Operons para Média, Desvio Padrão e Covariâncias

In [ ]:
modelos = {}  # Dicionário para armazenar os modelos


# Pega o melhor popsize em função do BIC
def get_popsize_by_bic(model):
    df = df_metrics_bic.loc[
        (df_metrics_bic["model"] == model) & (df_metrics_bic["col_x"] == col_x)
    ]
    return df["popsize"].values[0]


# 1. Treinar modelos para MÉDIA (4 modelos)
def treinar_medias(hyper, X_train_bins):
    print("\n[1/3] TREINANDO MODELOS DE MÉDIA")
    print("-" * 80)
    for largura in larguras:
        print(f"Treinando MÉDIA para {largura}...")
        y = train_por_bin[f"{largura}_mean"].values.astype(np.float64)
        # hyper["population_size"] = get_popsize_by_bic(f"{largura}_mean")
        modelo = p.treinar_operon(hyper, X_train_bins, y)
        modelos[f"{largura}_mean"] = modelo


# 2. Treinar modelos para DESVIO PADRÃO (4 modelos)
def treinar_stds(hyper, X_train_bins):
    print("\n[2/3] TREINANDO MODELOS DE DESVIO PADRÃO")
    print("-" * 80)
    for largura in larguras:
        print(f"Treinando DESVIO PADRÃO para {largura}...")
        y = train_por_bin[f"{largura}_std"].values.astype(np.float64)
        # hyper["population_size"] = get_popsize_by_bic(f"{largura}_std")
        modelo = p.treinar_operon(hyper, X_train_bins, y)
        modelos[f"{largura}_std"] = modelo


# 3. Treinar modelos para COVARIÂNCIAS (6 modelos)
def treinar_covs(hyper, X_train_bins):
    print("\n[3/3] TREINANDO MODELOS DE COVARIÂNCIA")
    print("-" * 80)
    for l1, l2 in p.cov_pairs:
        print(f"Treinando COVARIÂNCIA para {l1} x {l2}...")
        y = train_por_bin[f"cov_{l1}_{l2}"].values.astype(np.float64)
        # hyper["population_size"] = get_popsize_by_bic(f"cov_{l1}_{l2}")
        modelo = p.treinar_operon(hyper, X_train_bins, y)
        modelos[f"cov_{l1}_{l2}"] = modelo

In [ ]:
# Selecionar feature para o treinamento
X_train_bins = train_por_bin[col_x].values.reshape(-1, 1).astype(np.float64)

# Configuração do Operon
hyper = {
    "random_state": 4321,
    "population_size": 1000,
    "allowed_symbols": "add,sub,mul,aq,constant,variable,pow,exp,tanh",
    "max_length": 25,
    "max_depth": 25,
    "optimizer_iterations": 100,
    "model_selection_criterion": "bayesian_information_criterion",
    "objectives": ["r2", "length"],
    "n_threads": 12,
}

treinar_medias(hyper, X_train_bins)
treinar_stds(hyper, X_train_bins)
treinar_covs(hyper, X_train_bins)
p.salva_equacoes_html(modelos, col_x, f"n4d_{col_x}")

## Inspeção da qualidade dos modelos

In [ ]:
# p.inspecao_modelos(modelos, train_por_bin, col_x)
p.inspecao_modelos(modelos, test_por_bin, col_x)
plt.savefig(
    f"results/regressoes/inspecao_meanstds_n4d_{col_x}_test.png", bbox_inches="tight"
)

In [ ]:
# p.inspecao_modelos_covs(modelos, train_por_bin, col_x)
p.inspecao_modelos_covs(modelos, test_por_bin, col_x)
plt.savefig(
    f"results/regressoes/inspecao_covs_n4d_{col_x}_test.png", bbox_inches="tight"
)

In [ ]:
# # Exibir alguma função treinada de interesse
# x_pred = test[[col_x]]
# y_pred = modelos[f'cov_{T.ha.value}_{T.oiii.value}'].predict(x_pred)
# plt.scatter(x_pred, y_pred, s=1, alpha=0.5)
# plt.show()

## Gerar Amostras do Conjunto de Teste com Normal Multivariada

In [ ]:
def gerar_amostras(modelos, col_x):
    X_test = test[[col_x]].astype(np.float64)
    n_samples = len(X_test)
    print(
        f"Estimando parâmetros de {n_samples} pontos do conjunto de validação para a Normal Multivariada...\n"
    )
    means_all = np.column_stack(
        [modelos[f"{nome}_mean"].predict(X_test) for nome in larguras]
    )
    stds_all = np.column_stack(
        [np.maximum(modelos[f"{nome}_std"].predict(X_test), 1e-6) for nome in larguras]
    )
    covs_all = {}
    for l1, l2 in p.cov_pairs:
        covs_all[(l1, l2)] = modelos[f"cov_{l1}_{l2}"].predict(X_test)
    print("Predições dos estimadores concluídas!")

    n_correcoes = 0
    amostras_multivariadas = np.zeros((n_samples, 4))
    idx_map = {nome: j for j, nome in enumerate(larguras)}
    print("\nAmostragem multivariada iniciada..\n")

    for i in range(n_samples):
        if i % 5000 == 0 and i > 0:
            print(f"  Progresso: {i}/{n_samples} ({100*i/n_samples:.1f}%)")

        # Montar matriz de covariância
        mean_vector = means_all[i]
        stds = stds_all[i]
        cov_matrix = np.diag(stds**2)

        # Preencher covariâncias com validação
        for (l1_nome, l2_nome), cov_vals in covs_all.items():
            i1 = idx_map[l1_nome]
            i2 = idx_map[l2_nome]
            if np.isfinite(cov_vals[i]):
                cov_matrix[i1, i2] = cov_vals[i]
                cov_matrix[i2, i1] = cov_vals[i]

        # Remover infs e NaNs e garantir simetria
        cov_matrix = np.nan_to_num(cov_matrix, nan=0.0, posinf=0.0, neginf=0.0)
        # cov_matrix = (cov_matrix + cov_matrix.T) / 2

        # Regularização robusta
        corrigiu = False
        try:
            if not np.all(np.isfinite(cov_matrix)):
                raise ValueError("Matriz ainda contém valores não-finitos")

            # Corrigir autovalores negativos ou muito pequenos
            min_eigenval = 1e-8
            eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
            if np.any(eigenvalues < min_eigenval):
                eigenvalues = np.maximum(eigenvalues, min_eigenval)
                cov_matrix = eigenvectors @ np.diag(eigenvalues) @ eigenvectors.T
                corrigiu = True

            sample = multivariate_normal(
                mean=mean_vector, cov=cov_matrix, allow_singular=True
            ).rvs()

        except Exception as e:
            # Se tudo falhar, usar matriz diagonal
            cov_matrix = np.diag(stds**2)
            eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
            if not np.all(np.isfinite(cov_matrix)) or np.any(
                eigenvalues < min_eigenval
            ):
                corrigiu = False
                n_samples -= 1
                continue  # esquece e passa pra próxima combinação
            sample = multivariate_normal(
                mean=mean_vector, cov=cov_matrix, allow_singular=False
            ).rvs()
            corrigiu = True

        finally:
            if corrigiu:
                n_correcoes += 1

        amostras_multivariadas[i] = sample

    amostras = {
        col_x: X_test.values.flatten(),
        larguras[0]: amostras_multivariadas[:, 0],
        larguras[1]: amostras_multivariadas[:, 1],
        larguras[2]: amostras_multivariadas[:, 2],
        larguras[3]: amostras_multivariadas[:, 3],
    }

    print("\nAmostragem multivariada concluída!")
    print(f"   - Amostras: {n_samples}")
    print(
        f"   - Matrizes corrigidas: {n_correcoes} ({100*n_correcoes/n_samples:.1f}%)\n"
    )

    # Salvando amostras em CSV
    df_amostras = pd.DataFrame(amostras)
    df_amostras.to_csv(f"results/amostras_{col_x}.csv", index=False)

    print("Estatísticas:\n")
    for nome in larguras:
        valores = df_amostras[nome]
        print(
            f"  {nome:12s}: média={valores.mean():7.3f}, std={valores.std():6.3f}, "
            f"min={valores.min():7.3f}, max={valores.max():7.3f}"
        )
    return 100 * n_correcoes / n_samples


p_correcao = gerar_amostras(modelos, col_x)

In [ ]:
# Ler amostras geradas
df_amostras = pd.read_csv(f"results/amostras_{col_x}.csv")
df_amostras = df_amostras.dropna()
df_amostras = df_amostras.reset_index(drop=True)

# Calcular razões (já estão em log10, então é só subtrair)
test[T.nii_ha.value] = test[T.nii.value].values - test[T.ha.value].values
test[T.oiii_hb.value] = test[T.oiii.value].values - test[T.hb.value].values
df_amostras[T.nii_ha.value] = df_amostras[T.nii.value] - df_amostras[T.ha.value]
df_amostras[T.oiii_hb.value] = df_amostras[T.oiii.value] - df_amostras[T.hb.value]

df_amostras.tail()

### Histogramas marginais das amostras normais

In [ ]:
# Calcula divergência KL entre amostras e o conjunto observado
bins = np.linspace(-1, 2.5, 100)
kl_nii = p.calcula_kl(test, df_amostras, T.nii.value, bins)
kl_ha = p.calcula_kl(test, df_amostras, T.ha.value, bins)
kl_oiii = p.calcula_kl(test, df_amostras, T.oiii.value, bins)
kl_hb = p.calcula_kl(test, df_amostras, T.hb.value, bins)

# Gráficos
fig, axs = plt.subplots(2, 2, figsize=(16, 12))
plt.suptitle(
    "Distribuição das amostras normais geradas em função de " + p.unidades[col_x], fontsize="xx-large"
)
p.histogram_v(
    test[T.nii.value], "",
    axs[0][0], bins=bins, lim=(-1, 2.5),
    cor="darkblue", lbl="Test Set"    
)
p.histogram_v(
    test[T.ha.value], "",
    axs[0][1], bins=bins, lim=(-1, 2.5),
    cor="darkblue", lbl="Test Set"    
)
p.histogram_v(
    test[T.oiii.value], "",
    axs[1][0], bins=bins, lim=(-1, 2.5),
    cor="darkblue", lbl="Test Set"    
)
p.histogram_v(
    test[T.hb.value], "",
    axs[1][1], bins=bins, lim=(-1, 2.5),
    cor="darkblue", lbl="Test Set"    
)
p.histogram_v(
    df_amostras[T.nii.value],
    r"%s, $D_{KL}$ = %.4f" % (p.unidades[T.nii.value], kl_nii),
    axs[0][0], bins=bins, lim=(-1, 2.5),
    cor="lightgreen", lbl="Sample"
)
p.histogram_v(
    df_amostras[T.ha.value],
    r"%s, $D_{KL}$ = %.4f" % (p.unidades[T.ha.value], kl_ha),
    axs[0][1], bins=bins, lim=(-1, 2.5),
    cor="lightgreen", lbl="Sample"
)
p.histogram_v(
    df_amostras[T.oiii.value],
    r"%s, $D_{KL}$ = %.4f" % (p.unidades[T.oiii.value], kl_oiii),
    axs[1][0], bins=bins, lim=(-1, 2.5),
    cor="lightgreen", lbl="Sample"
)
p.histogram_v(
    df_amostras[T.hb.value],
    r"%s, $D_{KL}$ = %.4f" % (p.unidades[T.hb.value], kl_hb),
    axs[1][1], bins=bins, lim=(-1, 2.5),
    cor="lightgreen", lbl="Sample"
)
plt.tight_layout()
plt.savefig(
    f"results/histogramas/marginais_amostras_n4d_{col_x}_test.png", bbox_inches="tight"
)

## Calcular razões do BPT

In [ ]:
# Matriz de correlação das amostras geradas
corr_gerada = spearmanr(df_amostras[larguras]).correlation
df_corrger = pd.DataFrame(corr_gerada, index=larguras, columns=larguras)

# Matriz de correlação dos dados reais (test set)
test_linhas = test[larguras].values
corr_real = spearmanr(test_linhas).correlation
df_corrtest = pd.DataFrame(corr_real, index=larguras, columns=larguras)

# Diferença total absoluta
diff_df = (df_corrger - df_corrtest).abs()
diff = diff_df.values[np.triu_indices(4, k=1)].sum()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
p.plot_corr(
    corr_real,
    axes[0],
    larguras,
    title="Matriz de correlação dos dados de teste",
    type="inf",
)
p.plot_corr(
    corr_gerada,
    axes[1],
    larguras,
    title="Matriz de correlação das amostras normais\n"
    + r"Correções: %.1f%%, $\Delta$=%.4f" % (p_correcao, diff),
    type="inf",
)
plt.savefig(
    f"results/correlacoes/corr_amostras_n4d_{col_x}_test.png", bbox_inches="tight"
)

### Diagramas de diagnóstico

In [ ]:
# Plotar com curvas de densidade
p.show_bpt(df_amostras, title="Estimadores Operon + Amostras Normal4D", densities=True, show=False)
plt.savefig(f"results/diagramas/bpt_densidades_amostras_n4d_{col_x}_test.png", bbox_inches="tight")

In [ ]:
# Plotar amostras geradas coloridas por alguma feature
p.show_bpt(df_amostras, col_x, title="Estimadores Operon + Amostras Normal4D", show=False)
plt.savefig(f"results/diagramas/bpt_cores_amostras_n4d_{col_x}_test.png", bbox_inches="tight")

In [ ]:
# Plotar com curvas de densidade
p.show_whan(df_amostras, title="Estimadores Operon + Amostras Normal4D", densities=True, show=False)
plt.savefig(f"results/diagramas/whan_densidades_amostras_n4d_{col_x}_test.png", bbox_inches="tight")

In [ ]:
p.show_whan(df_amostras, col_x, title="Estimadores Operon + Amostras Normal4D", show=False)
plt.savefig(f"results/diagramas/whan_cores_amostras_n4d_{col_x}_test.png", bbox_inches="tight")